<a href="https://colab.research.google.com/github/nejumi/weave-initial-course/blob/main/notebooks/01_basics/02_assets.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02. アセット管理：Model / Prompt / Dataset

Weave では、モデル・プロンプト・データセットを「アセット」として一元管理・バージョン付けできます。

## 学習内容
1. `weave.Model` — LLM ラッパーのバージョン管理
2. `weave.StringPrompt` / `weave.MessagesPrompt` — プロンプトのバージョン管理
3. `weave.Dataset` — データセットの作成・取得・変換


In [ ]:
# ローカル環境: uv sync --all-extras で一括インストール可能
# Colab: 以下のセルで個別インストール
!pip install -q uv
!uv pip install --system -q \
  "weave>=0.51.0" \
  "wandb>=0.19.0" \
  "openai>=1.0.0" \
  "python-dotenv>=1.0.0" \
  "datasets>=2.0.0" \
  "pandas>=2.0.0"

In [ ]:
import os

# Colab / ローカル環境の自動判定
try:
    import google.colab
    IN_COLAB = True
    from google.colab import userdata
    # WANDB_BASE_URL: 値がある場合のみセット（空 / 未設定 → SaaS デフォルト）
    _base_url = (userdata.get("WANDB_BASE_URL") or "").strip()
    if _base_url:
        os.environ["WANDB_BASE_URL"] = _base_url
    os.environ.setdefault("WANDB_API_KEY",  userdata.get("WANDB_API_KEY"))
    os.environ.setdefault("OPENAI_API_KEY", userdata.get("OPENAI_API_KEY"))
    os.environ.setdefault("WANDB_ENTITY",   userdata.get("WANDB_ENTITY"))
    os.environ.setdefault("WANDB_PROJECT",  userdata.get("WANDB_PROJECT") or "weave-course")
    print("Google Colab 環境")
except ImportError:
    IN_COLAB = False
    from dotenv import load_dotenv
    load_dotenv()
    # ローカル: .env に WANDB_BASE_URL= と書かれていた場合も空なら削除
    if not os.environ.get("WANDB_BASE_URL", "").strip():
        os.environ.pop("WANDB_BASE_URL", None)
    print("ローカル環境")

ENTITY  = os.environ["WANDB_ENTITY"]
PROJECT = os.environ.get("WANDB_PROJECT", "weave-course")
_target = os.environ.get("WANDB_BASE_URL", "https://api.wandb.ai (SaaS)")
print(f"Entity : {ENTITY}")
print(f"Project: {PROJECT}")
print(f"W&B URL: {_target}")


In [ ]:
import weave
client = weave.init(f"{ENTITY}/{PROJECT}")

## 1. `weave.Model`

`weave.Model` を継承し、`predict` に `@weave.op()` をつけることで、
モデルの設定（system_message, temperature など）と推論コードが一体となってバージョン管理されます。


In [ ]:
from openai import OpenAI

class GrammarCorrector(weave.Model):
    system_message: str
    model_name: str
    temperature: float = 0.0

    @weave.op()
    def predict(self, sentence: str) -> dict:
        oai = OpenAI()
        resp = oai.chat.completions.create(
            model=self.model_name,
            messages=[
                {"role": "system", "content": self.system_message},
                {"role": "user",   "content": sentence},
            ],
            temperature=self.temperature,
        )
        return {"corrected": resp.choices[0].message.content}

# バージョン1: predict を呼ぶと Weave UI に GrammarCorrector:v0 として自動登録される
model_v1 = GrammarCorrector(
    system_message="You are a grammar checker. Return only the corrected sentence.",
    model_name="gpt-5.4-mini-2026-03-17",
)
result = model_v1.predict("She goed to the store yesterday.")
print("v1 result:", result)
# → Weave UI で Objects > GrammarCorrector:v0 を確認してください

In [ ]:
# system_message を変えて predict → GrammarCorrector:v1 が自動作成される
model_v2 = GrammarCorrector(
    system_message="You are an expert grammar checker. Return only the corrected sentence, no explanation.",
    model_name="gpt-5.4-mini-2026-03-17",
)
result = model_v2.predict("I has a big dog.")
print("v2 result:", result)
# → Weave UI で GrammarCorrector:v1 が追加されたことを確認してください

# weave.publish で明示的に名前をつけて参照できるようにすることも可能
# （Prompt や Dataset の管理と同じ API）
ref = weave.publish(model_v2, name="grammar_corrector")
print("published ref:", ref)

In [ ]:
# ref で特定バージョンを取得
retrieved = weave.ref(f"weave:///{ENTITY}/{PROJECT}/object/grammar_corrector:latest").get()
print(f"取得したモデル: system_message={retrieved.system_message[:50]}...")

## 2. `weave.StringPrompt` / `weave.MessagesPrompt`

プロンプトを独立したアセットとして管理することで、
コードとプロンプトのバージョンを分離して追跡できます。


In [ ]:
# StringPrompt — 単一文字列プロンプト
system_prompt = weave.StringPrompt("You speak like a friendly pirate. 🏴‍☠️")
try:
    weave.publish(system_prompt, name="pirate_prompt")
except Exception:
    pass

oai = OpenAI()
resp = oai.chat.completions.create(
    model="gpt-5.4-mini-2026-03-17",
    messages=[
        {"role": "system", "content": system_prompt.format()},
        {"role": "user", "content": "What is machine learning?"},
    ],
    max_completion_tokens=80,
)
print(resp.choices[0].message.content)


In [ ]:
# MessagesPrompt — 会話履歴形式
messages_prompt = weave.MessagesPrompt([
    {"role": "system", "content": "You are a helpful assistant specializing in {domain}."},
    {"role": "user", "content": "{question}"},
])
try:
    weave.publish(messages_prompt, name="domain_expert_prompt")
except Exception:
    pass

# テンプレート変数を埋め込んで使用
formatted = messages_prompt.format(domain="machine learning", question="What is overfitting?")
resp = oai.chat.completions.create(model="gpt-5.4-mini-2026-03-17", messages=formatted, max_completion_tokens=80)
print(resp.choices[0].message.content)


## 3. `weave.Dataset`

評価用サンプルを整理・収集・追跡するための Weave オブジェクトです。
Hugging Face Datasets に近いインターフェースです。


In [ ]:
from weave import Dataset

# rows から作成
dataset = Dataset(
    name="grammar_benchmark",
    rows=[
        {"id": "0", "sentence": "He no likes ice cream.",   "correction": "He doesn't like ice cream."},
        {"id": "1", "sentence": "She goed to the store.",    "correction": "She went to the store."},
        {"id": "2", "sentence": "They was playing outside.", "correction": "They were playing outside."},
        {"id": "3", "sentence": "I has a big dog.",          "correction": "I have a big dog."},
        {"id": "4", "sentence": "We runned very fast.",      "correction": "We ran very fast."},
    ],
)
weave.publish(dataset)
print(f"データセット公開完了: {len(dataset.rows)} 件")


In [ ]:
# Pandas DataFrame から作成
import pandas as pd

df = pd.DataFrame([
    {"question": "What is Weave?",   "answer": "An LLM observability platform by W&B."},
    {"question": "What is @weave.op?", "answer": "A decorator that traces function calls."},
])
ds_from_df = Dataset.from_pandas(df)
weave.publish(ds_from_df, name="faq_dataset")
print(ds_from_df.to_pandas())


In [ ]:
# Weave Call から作成
@weave.op()
def dummy_model(sentence: str) -> str:
    return sentence.upper()

_, c1 = dummy_model.call("hello world")
_, c2 = dummy_model.call("weave is great")

ds_from_calls = Dataset.from_calls([c1, c2])
weave.publish(ds_from_calls, name="calls_dataset")
print(f"Call から作成: {len(ds_from_calls.rows)} 件")


In [ ]:
# バージョン指定で取得
retrieved_ds = weave.ref(f"weave:///{ENTITY}/{PROJECT}/object/grammar_benchmark:latest").get()
print(f"取得したデータセット: {len(retrieved_ds.rows)} 件")
print("最初の行:", dict(retrieved_ds.rows[0]))


## まとめ

次: **03_models.ipynb** → `weave.Model` の深掘り